In [ ]:

!pip install -q --upgrade pip
!pip install -q pandas scikit-learn sentence-transformers tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.8 MB/s eta 0:00:00


In [ ]:

import os
import pickle
from tqdm import tqdm
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# For SBERT
from sentence_transformers import SentenceTransformer

# Config
DATA_PATH = "/content/sample_data/partner_dataset_3000.csv"  # change if you uploaded to a different path
OUTPUT_PKL = "/content/sample_data/partner_dataset_3000.pkl"
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [ ]:
pip install triton

In [ ]:

df = pd.read_csv(DATA_PATH)
print("Rows:", len(df))
df.head(3)


Rows: 3000


,_id,orgName,email,role,location,country,description,interests,skills,website,contact_person,phone,capacity_score,avg_project_budget,active_projects,languages,timezone,created_at
0,partner_0001,ImpactNetwork 1 (Government),contact1@impactnetwork.org,Government,"London, UK",UK,CommunityWorks 1 (Government) focuses on gis a...,SDG 9,"Communications, Project Management, Mobile App...",https://www.impactnetwork1.org,Lead 1,44892891174,95,10000,12,"English, French",UTC,2025-10-23T06:12:22.380407Z
1,partner_0002,ImpactInitiative 2 (Startup),contact2@impactinitiativ.org,Startup,"Tokyo, Japan",Japan,UrbanPartners 2 (Startup) focuses on gis and w...,SDG 10,"Water Management, Mobile App Development",https://www.impactinitiativ2.org,Lead 2,2033246548,80,10000,2,"English, Arabic",Asia/Tokyo,2024-07-06T06:12:22.380407Z
2,partner_0003,ImpactWorks 3 (University),contact3@impactworks.org,University,"Bangkok, Thailand",Thailand,RuralFoundation 3 (University) focuses on agri...,"SDG 11, SDG 6, SDG 14, SDG 5","Research, Agriculture",https://www.impactworks3.org,Lead 3,1658900037,70,100000,5,"English, Arabic",UTC,2025-07-12T06:12:22.380407Z


In [ ]:

expected = ['_id','orgName','description','interests','skills','role','location','capacity_score','avg_project_budget','active_projects']
print("Columns present:", df.columns.tolist())
for c in expected:
    print(c, "->", c in df.columns)


Columns present: ['_id', 'orgName', 'email', 'role', 'location', 'country', 'description', 'interests', 'skills', 'website', 'contact_person', 'phone', 'capacity_score', 'avg_project_budget', 'active_projects', 'languages', 'timezone', 'created_at']
_id -> True
orgName -> True
description -> True
interests -> True
skills -> True
role -> True
location -> True
capacity_score -> True
avg_project_budget -> True
active_projects -> True


In [ ]:
# Cleaning
import re
from sklearn.utils import shuffle

def clean_text(s):
    if not isinstance(s, str): return ""
    s = s.replace("\n", " ").replace("\r", " ")
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def parse_interests(s):
    if pd.isna(s): return []
    if isinstance(s, list): return [re.sub(r'[^0-9]','',str(x)).strip() for x in s if re.sub(r'[^0-9]','',str(x)).strip()]
    # possible forms: "SDG 6, SDG 14" or "6,14"
    parts = re.split(r'[;,|/]', str(s))
    out = []
    for p in parts:
        p = p.strip()
        m = re.search(r'(\d{1,2})', p)
        if m:
            out.append(m.group(1))
    return list(dict.fromkeys(out))  # unique preserve order

# Clean columns
df['_id'] = df.get('_id', pd.Series([f"partner_{i}" for i in range(len(df))]))
df['orgName'] = df['orgName'].fillna('Unknown Org').astype(str).map(clean_text)
df['description'] = df['description'].fillna('').astype(str).map(clean_text)
df['skills'] = df['skills'].fillna('').astype(str).map(clean_text)
df['location'] = df['location'].fillna('').astype(str).map(clean_text)
df['interests_list'] = df['interests'].apply(parse_interests)

# numeric fallbacks
df['capacity_score'] = pd.to_numeric(df.get('capacity_score', 50), errors='coerce').fillna(50)
df['avg_project_budget'] = pd.to_numeric(df.get('avg_project_budget', 10000), errors='coerce').fillna(10000)
df['active_projects'] = pd.to_numeric(df.get('active_projects', 0), errors='coerce').fillna(0)

# Create a combined text for vectorizers
def make_text_row(r):
    parts = [
        r.get('orgName',''),
        r.get('description',''),
        r.get('skills',''),
        r.get('location',''),
        " ".join([f"SDG{n}" for n in r.get('interests_list',[])])
    ]
    return " | ".join([p for p in parts if p])

df['text'] = df.apply(make_text_row, axis=1)

# Shuffle dataset for robustness
df = shuffle(df, random_state=RANDOM_SEED).reset_index(drop=True)

print("Sample cleaned row:")
display(df[[' _id' if ' _id' in df.columns else '_id','orgName','interests_list','text']].head(2))


Sample cleaned row:


,_id,orgName,interests_list,text
0,partner_1802,RuralWorks 1802 (NGO),"[8, 3]",RuralWorks 1802 (NGO) | FutureFoundation 1802 ...
1,partner_1191,RuralInitiative 1191 (Private Sector),"[4, 14, 5]",RuralInitiative 1191 (Private Sector) | SolarS...


In [ ]:
# Baseline pipelines
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2), stop_words='english')
X_tfidf = tfidf.fit_transform(df['text'].values)

# Fit KNN on TF-IDF with cosine distance
knn_tfidf = NearestNeighbors(n_neighbors=11, metric='cosine', algorithm='brute')  # n_neighbors includes self
knn_tfidf.fit(X_tfidf)
print("TF-IDF shape:", X_tfidf.shape)


TF-IDF shape: (3000, 13938)


In [ ]:
#Recommended pipeline: SBERT embeddings + KNN
model_name = "all-MiniLM-L6-v2"
print("Loading SBERT:", model_name)
sbert = SentenceTransformer(model_name)

# compute embeddings
texts = df['text'].tolist()
embeddings = sbert.encode(texts, show_progress_bar=True, convert_to_numpy=True)
print("Embeddings shape:", embeddings.shape)

# Normalize embeddings (recommended for cosine)
from sklearn.preprocessing import normalize
embeddings_norm = normalize(embeddings, norm='l2')

# Fit KNN (cosine via metric='cosine')
knn_sbert = NearestNeighbors(n_neighbors=11, metric='cosine', algorithm='auto')
knn_sbert.fit(embeddings_norm)

Loading SBERT: all-MiniLM-L6-v2


Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Embeddings shape: (3000, 384)


NearestNeighbors(metric='cosine', n_neighbors=11)

In [ ]:
def precision_at_k(knn_model, vectors, partner_sdg_lists, k=5):
    # returns avg precision@k (excluding self match)
    n = vectors.shape[0]
    total_p = 0.0
    for i in range(n):
        dist, inds = knn_model.kneighbors(vectors[i].reshape(1, -1), n_neighbors=k+1, return_distance=True)
        inds = inds[0].tolist()
        # remove self (i)
        inds = [j for j in inds if j != i][:k]
        hits = 0
        q_sdg = set(partner_sdg_lists[i])
        if len(q_sdg) == 0:
            continue
        for j in inds:
            if q_sdg.intersection(partner_sdg_lists[j]):
                hits += 1
        total_p += hits / k
    return total_p / n

# Prepare sdg lists for evaluation
sdg_lists = [set(x) for x in df['interests_list'].tolist()]

# For TF-IDF we need vector rows; X_tfidf is sparse
print("Evaluating TF-IDF KNN")
p_tfidf = precision_at_k(knn_tfidf, X_tfidf, sdg_lists, k=5)
print("TF-IDF precision@5:", round(p_tfidf,4))

print("Evaluating SBERT KNN")
p_sbert = precision_at_k(knn_sbert, embeddings_norm, sdg_lists, k=5)
print("SBERT precision@5:", round(p_sbert,4))


Evaluating TF-IDF KNN
TF-IDF precision@5: 0.5805
Evaluating SBERT KNN
SBERT precision@5: 0.4688


In [ ]:
# Colab cell - choose final pipeline (recommend SBERT)
final_dict = {
    'model_type': 'sbert',
    'rec_model': knn_sbert,
    'vectorizer': None,  # tfidf if you'd like baseline
    'embeddings': embeddings_norm,
    'partner_ids': df['_id'].tolist(),
    'partners_df': df,   # pandas DF (serialize-able)
    'sbert_model_name': model_name
}

with open(OUTPUT_PKL, 'wb') as f:
    pickle.dump(final_dict, f)

print("Saved recommendation artifact to:", OUTPUT_PKL)


Saved recommendation artifact to: /content/sample_data/partner_dataset_3000.pkl


In [ ]:
from google.colab import files
files.download(OUTPUT_PKL)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>